# LlamaIndex Episode 1 🦙

## Overview

* What is LlamaIndex?

        * LlamaHub (data loaders)

* How to setup Weaviate

        * Create schema


* Adding Data to Weaviate using LlamaIndex

        *  Data loader examples

* Chunking up your data

* Connecting Weaviate instance to LlamaIndex

* Simple query engine

## What is [LlamaIndex](https://www.llamaindex.ai/)?

#### Framework that enables you to connect LLMs and storage providers together seamlessly.
#### LlamaIndex 🤝 Weaviate ➡ Ultimate RAG stack

#### [LlamaHub](https://llama-hub-ui.vercel.app/): Enables you to connect to a number of external data sources (Notion, Slack, Web pages, and more!)

## Setting up Weaviate

We first first to initialize a Weaviate client and hand it over to LlamaIndex. You can do that in different ways:

1. Embedded - Runs a local Weaviate cluster. Works on Linux and Mac.

2. WCD - Connects to Weaviate Cloud. You can spin up a free sandbox cluster at https://console.weaviate.cloud/ and get the url and api key.

3. Docker - run in Docker. You can use our [docker configurator tool](https://weaviate.io/developers/weaviate/installation/docker-compose#configurator) to get you started.

Let's first install weaviate client and llama-index along some other dependencies:

In [ ]:
%pip install -U weaviate-client llama-index llama-index-vector-stores-weaviate llama-index-embeddings-openai

In [ ]:
# let's catch some logs
import logging
import sys

logging.basicConfig(stream=sys.stdout, level=logging.INFO)
logging.getLogger().addHandler(logging.StreamHandler(stream=sys.stdout))

### Embedded

In [ ]:
import weaviate

client = weaviate.connect_to_embedded()

### WCD

In [ ]:
import weaviate
import os
  
# Set these environment variables
URL = os.getenv("WCD_URL")
APIKEY = os.getenv("WCD_API_KEY")
  
# Connect to a WCD instance
client = weaviate.connect_to_weaviate_cloud(
    cluster_url=URL,
    auth_credentials=weaviate.auth.AuthApiKey(APIKEY)
)

### Docker

In order to run with docker, you can use our [docker configurator tool](https://weaviate.io/developers/weaviate/installation/docker-compose#configurator). 

Once you have Weaviate running with docker, you can get the client with:

In [ ]:
import weaviate
from weaviate import classes as wvc
  
# Connect to a local instance
client = weaviate.connect_to_local()

### Collection
Let's create our collection before hand, and specify a model to use. This model must be the same one used in LlamaIndex.

In [ ]:
from weaviate import classes as wvc
# clean slate
client.collections.delete("BlogPost")

collection = client.collections.create(
    name="BlogPost",
    description="Blog post from the Weaviate website.",
    vectorizer_config=wvc.config.Configure.Vectorizer.text2vec_openai(
        model="text-embedding-3-small"
    ),
    generative_config=wvc.config.Configure.Generative.openai(
        model="gpt-3.5-turbo"
    ),
    properties=[
        wvc.config.Property(name="text", description="Content from the blog post", data_type=wvc.config.DataType.TEXT)
    ]
)

print("Collection was created.")

## Adding Data to Weaviate using LlamaIndex

### SimpleDirectoryReader: Read files in your filesystem

In [ ]:
from llama_index.core import SimpleDirectoryReader

documents = SimpleDirectoryReader('./data').load_data()

### SimpleWebPageReader: Web scraper that turns HTML to text

In [ ]:
from llama_index.readers.web import SimpleWebPageReader

loader = SimpleWebPageReader()
documents = loader.load_data(urls=['https://weaviate.io/blog/llamaindex-and-weaviate'])

### NotionPageReader: Loads documents from Notion

In [ ]:
%pip install llama-index-readers-notion

In [ ]:
from llama_index.readers.notion import NotionPageReader

integration_token = ("secret_key")
page_ids = ["40be241cac924a5aa887fa85e945dbf6"]
reader = NotionPageReader(integration_token=integration_token)
documents = reader.load_data(page_ids=page_ids)

### Inspecting the nodes of you documents

In [ ]:
from llama_index.core.node_parser import SimpleNodeParser

parser = SimpleNodeParser()
nodes = parser.get_nodes_from_documents(documents)
print("Number of nodes:", len(nodes))
print(nodes[0])

### Documents to Weaviate

In [ ]:
from llama_index.vector_stores.weaviate import WeaviateVectorStore
from llama_index.core import VectorStoreIndex, StorageContext, Settings
from llama_index.embeddings.openai import OpenAIEmbedding
import openai
import os

# global
Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small")

# Lets set the OPENAI key
# os.environ["OPENAI_API_KEY"] = "sk-key"
openai.api_key = os.environ["OPENAI_API_KEY"]

# loading the documents
documents = SimpleDirectoryReader("./data/").load_data()

# Let's name our index properly as BlogPost, as we will need it later.
vector_store = WeaviateVectorStore(
    weaviate_client=client, index_name="BlogPost"
)
storage_context = StorageContext.from_defaults(vector_store=vector_store)
index = VectorStoreIndex.from_documents(
    documents, storage_context=storage_context
)

In [ ]:
# Let's check if the objects were created
collection = client.collections.get("BlogPost")
query = collection.query.fetch_objects()
if query.objects:
    print("Objects in this collection:", len(collection))
    print("Object properties example:", query.objects[0].properties)
else:
    print("No objects found in this collection.")

### Query in LlamaIndex

In [ ]:
from llama_index.vector_stores.weaviate import WeaviateVectorStore
from llama_index.core import VectorStoreIndex

vector_store = WeaviateVectorStore(
    weaviate_client=client, index_name="BlogPost"
)

loaded_index = VectorStoreIndex.from_vector_store(vector_store)

query_engine = loaded_index.as_query_engine()
response = query_engine.query("What is the intersection between LLMs and search?")
print(response)